# 🎫 Notebook 3 — Top-k: The VIP List

> **The vibe:** A nightclub bouncer. Only the top-k ranked tokens get past the velvet rope. Doesn't matter how cool token #k+1 is — if you're not on the list, you're not getting in.  
> **Key word:** FIXED — the guest list never changes size, no matter what.  
> **Time:** ~10 minutes


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','text.color':'#e6edf3','axes.labelcolor':'#8b949e',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','grid.color':'#21262d',
    'axes.grid':True,'grid.linewidth':0.5,'font.family':'monospace'})

TOKENS = ['approve','reject','review','escalate','delay','audit','optimize','notify','assign','close']
LOGITS = np.array([2.2,1.8,1.4,0.9,0.2,0.1,-0.3,-0.6,-0.8,-1.0])

def softmax(x): e=np.exp(x-x.max()); return e/e.sum()
def apply_top_k(probs, k):
    if k <= 0 or k >= len(probs): return probs.copy()
    filtered = np.zeros_like(probs)
    top_idx  = np.argsort(probs)[-k:]
    filtered[top_idx] = probs[top_idx]
    return filtered / filtered.sum()

def apply_top_p(probs, p):
    sorted_idx = np.argsort(probs)[::-1]
    cumsum = np.cumsum(probs[sorted_idx])
    cutoff = np.searchsorted(cumsum, p) + 1
    nucleus_idx = sorted_idx[:cutoff]
    filtered = np.zeros_like(probs)
    filtered[nucleus_idx] = probs[nucleus_idx]
    return filtered / filtered.sum(), cutoff

PROBS = softmax(LOGITS)
print("✅ Setup done!")
print("\n🎫 The VIP Rankings:")
for rank, (tok, p) in enumerate(sorted(zip(TOKENS, PROBS), key=lambda x: -x[1]), 1):
    vip = "🟢 VIP" if rank <= 3 else "🔴 NO ENTRY" if rank > 5 else "🟡 maybe"
    print(f"  Rank {rank:2d}: {tok:12s} ({p:.3f})  {vip}")


## 🎬 Graph 1 — The Velvet Rope at Different k Values

Watch the bouncer's list change: k=1 is ultra-exclusive, k=10 lets everyone in.

**Key observation:** Count the colored bars in each panel — that's always exactly k.


In [ ]:
# ── Graph 1: The Velvet Rope ─────────────────────────────────────────
k_values = [1, 3, 5, 7, 10]
k_colors = ['#ff6b9d','#ff9f43','#ffd93d','#a8e6cf','#74b9ff']
k_labels = ['🚫 k=1\nGreedy', '🌟 k=3\nStrict', '✅ k=5\nModerate',
            '😊 k=7\nOpen', '🎉 k=10\nAll in!']

sorted_idx  = np.argsort(PROBS)[::-1]
sorted_prob = PROBS[sorted_idx]
sorted_name = [TOKENS[i] for i in sorted_idx]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('🎫  Top-k: The Velvet Rope — k = How Many Get Past the Bouncer',
             fontsize=13, fontweight='bold', color='#ffd93d', y=1.02)

for ax, k, col, lbl in zip(axes, k_values, k_colors, k_labels):
    filtered = apply_top_k(PROBS, k)
    filt_sorted = filtered[sorted_idx]
    bar_colors = [col if filt_sorted[i] > 0 else '#2d333b' for i in range(10)]
    ax.bar(range(10), filt_sorted, color=bar_colors, width=0.7, alpha=0.9)
    ax.axvline(k - 0.5, color='white', lw=2.5, ls='--')
    ax.set_xticks(range(10))
    ax.set_xticklabels(sorted_name, rotation=45, ha='right', fontsize=7)
    ax.set_ylim(0, max(sorted_prob)*1.2)
    ax.set_ylabel('Probability' if ax==axes[0] else '')
    entropy = -np.sum([p*np.log(p+1e-12) for p in filt_sorted if p > 0])
    ax.set_title(f'{lbl}\nH={entropy:.2f} bits', color=col, fontweight='bold', fontsize=10)
    ax.text(k-1, filt_sorted[k-1]+0.01, '⬆VIP', ha='center', va='bottom', color=col, fontsize=7)
    if k < 10:
        ax.text(k, sorted_prob[k]*0.5, '⛔', ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.savefig('graph1_topk_rope.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 Notice: exactly k bars are colored in each panel.")
print("💡 The '⛔' marks the first token that got REJECTED regardless of its probability.")


## 🎬 Graph 2 — Where Top-k Fails: The Mismatch Problem

Top-k's fixed size is both its strength and weakness.

- **Peaked distribution**: k=5 keeps 3 "nearly useless" tokens that barely matter  
- **Flat distribution**: k=5 cuts 5 tokens that have meaningful probability

**This is why top-p usually wins.** Top-p's nucleus adapts. Top-k's doesn't.


In [ ]:
# ── Graph 2: The Mismatch Problem ───────────────────────────────────
probs_peaked = softmax(np.array([4.0, 0.5, 0.2, 0.1,-0.1,-0.3,-0.5,-0.7,-0.9,-1.1]))
probs_flat   = softmax(np.array([0.5, 0.4, 0.3, 0.2, 0.1, 0.0,-0.1,-0.2,-0.3,-0.4]))

K_FIXED = 5

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'⚠️  Top-k={K_FIXED}: The Same Rule, Two Different Failures',
             fontsize=14, fontweight='bold', color='#ffd93d')

for row, (probs_ex, title, col) in enumerate([
    (probs_peaked, f'😎 CONFIDENT Model
(top-k={K_FIXED} keeps "junk" tokens)', '#ff6b9d'),
    (probs_flat,   f'🤔 UNCERTAIN Model
(top-k={K_FIXED} cuts valid tokens!)', '#74b9ff')
]):
    sorted_idx2 = np.argsort(probs_ex)[::-1]
    sorted_prob2 = probs_ex[sorted_idx2]
    filtered_k  = apply_top_k(probs_ex, K_FIXED)
    _, p90_size = apply_top_p(probs_ex, 0.9)

    # Panel 1: Top-k result
    bar_c = ['#ffd93d' if i < K_FIXED else '#2d333b' for i in range(10)]
    axes[row][0].bar(range(10), sorted_prob2, color=bar_c, width=0.7, alpha=0.9)
    axes[row][0].axvline(K_FIXED-0.5, color='#ffd93d', lw=2.5, ls='--', label=f'k={K_FIXED} cutoff')
    axes[row][0].axvline(p90_size-0.5, color='#ff6b9d', lw=2, ls=':', label=f'p=0.9 would cut here ({p90_size} tokens)')
    axes[row][0].set_title(title, color=col, fontweight='bold')
    axes[row][0].set_xlabel('Rank'); axes[row][0].set_ylabel('Probability')
    axes[row][0].legend(fontsize=8)

    # Panel 2: Wasted / Missing tokens
    waste_msg = f"TOP-K WASTES {K_FIXED - p90_size} TOKENS
(included but barely matter)"                 if K_FIXED > p90_size else                 f"TOP-K MISSES {p90_size - K_FIXED} TOKENS
(excluded but meaningful!)"
    waste_col = '#ff9f43' if K_FIXED > p90_size else '#ff6b9d'
    axes[row][1].axis('off')
    axes[row][1].text(0.5, 0.5, waste_msg,
        ha='center', va='center', transform=axes[row][1].transAxes,
        fontsize=14, color=waste_col, fontweight='bold',
        bbox=dict(boxstyle='round,pad=1', facecolor='#21262d', edgecolor=waste_col, lw=2))

    # Panel 3: Comparison bars
    top5_k  = sorted_prob2[:K_FIXED].sum()
    top5_p9 = sorted_prob2[:p90_size].sum()
    axes[row][2].bar(['top-k covers', 'p=0.9 covers'], [top5_k, top5_p9],
                    color=[col, '#a8e6cf'], alpha=0.85, width=0.5)
    axes[row][2].axhline(0.9, color='white', lw=1.5, ls='--', label='90% target')
    axes[row][2].set_ylabel('Probability mass covered')
    axes[row][2].set_title(f'Coverage Comparison
(both trying to cover ~90%)', color=col, fontweight='bold')
    axes[row][2].legend(fontsize=9); axes[row][2].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('graph2_topk_mismatch.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n💡 For the CONFIDENT model: k={K_FIXED} wastes slots on near-zero tokens.")
print(f"💡 For the UNCERTAIN model: k={K_FIXED} unfairly cuts meaningful tokens.")
print("💡 Top-p handles BOTH cases correctly with the same p=0.9!")


## 🎬 Graph 3 — Entropy vs k (The Diminishing Returns Curve)

Adding more tokens to the guest list increases diversity — but with diminishing returns.

Going from k=1 to k=2 is a HUGE jump.  
Going from k=8 to k=10 barely matters.

**Find the "elbow" — that's your optimal k for this distribution.**


In [ ]:
# ── Graph 3: Entropy vs k ────────────────────────────────────────────
k_range = range(1, 11)
entropies_k = []
for k in k_range:
    filt = apply_top_k(PROBS, k)
    e = -np.sum(filt * np.log(filt + 1e-12))
    entropies_k.append(e)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📈  Entropy vs k: Where Does Adding Tokens Stop Helping?',
             fontsize=14, fontweight='bold', color='#ffd93d')

ax1.plot(list(k_range), entropies_k, color='#a8e6cf', lw=2.5, marker='o', ms=8)
ax1.fill_between(list(k_range), entropies_k, alpha=0.15, color='#a8e6cf')
# Find the elbow
diffs = np.diff(entropies_k)
elbow = np.argmax(diffs < diffs[0] * 0.2) + 1
ax1.axvline(elbow, color='#ffd93d', lw=2, ls='--', label=f'Elbow ≈ k={elbow}')
ax1.annotate(f'Big jump here!
Each new token
still matters a lot',
             (2, entropies_k[1]), xytext=(3.5, entropies_k[1]-0.3),
             color='#ff9f43', fontsize=9,
             arrowprops=dict(arrowstyle='->', color='#ff9f43'))
ax1.annotate(f'Diminishing returns
(adding more k
bardly changes things)',
             (8, entropies_k[7]), xytext=(5.5, entropies_k[7]+0.3),
             color='#8b949e', fontsize=9,
             arrowprops=dict(arrowstyle='->', color='#8b949e'))
ax1.set_xlabel('k value'); ax1.set_ylabel('Entropy (bits)')
ax1.set_title('Entropy rises with k, but flattens out', color='#a8e6cf', fontweight='bold')
ax1.legend(fontsize=10); ax1.set_xticks(range(1, 11))

# Probability mass table
ranks = list(range(1, 11))
probs_sorted = np.sort(PROBS)[::-1]
cum_probs = np.cumsum(probs_sorted)
colors_bar = ['#ff6b9d' if c < 0.5 else '#ff9f43' if c < 0.8 else '#a8e6cf' if c < 0.95 else '#74b9ff'
              for c in cum_probs]
ax2.bar(ranks, probs_sorted, color=colors_bar, width=0.7, alpha=0.9)
ax2_twin = ax2.twinx()
ax2_twin.plot(ranks, cum_probs, color='white', lw=2, ls='--', marker='s', ms=5)
ax2_twin.axhline(0.9, color='#ffd93d', lw=1.5, ls=':', label='90% line')
ax2_twin.set_ylabel('Cumulative probability', color='white')
ax2_twin.tick_params(colors='white')
ax2_twin.legend(fontsize=9)
ax2.set_xlabel('Rank'); ax2.set_ylabel('Probability')
ax2.set_title('Probability mass by rank\n(cumulative = white dashed)',
              color='#ffd93d', fontweight='bold')
ax2.set_xticks(ranks)
ax2.set_xticklabels([f'#{r}\n{sorted(TOKENS, key=lambda t: -PROBS[TOKENS.index(t)])[r-1]}' for r in ranks],
                    fontsize=7)

plt.tight_layout()
plt.savefig('graph3_topk_entropy.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n💡 The elbow is around k={elbow} for this distribution.")
print("💡 Below the elbow: each extra token adds meaningful diversity.")
print("💡 Above the elbow: adding tokens barely changes behavior.")


## 🎮 Graph 4 — Playground: Move the Rope

Change `K_VALUE` to any number 1–10 and re-run.

Also try: compare top-k vs top-p by uncommenting the `COMPARE_P` line!


In [ ]:
# ── Graph 4: Playground ──────────────────────────────────────────────
K_VALUE   = 5    # ← 🎮 CHANGE THIS! Try: 1, 2, 3, 5, 8, 10
COMPARE_P = 0.9  # ← top-p value to compare against

filtered_k      = apply_top_k(PROBS, K_VALUE)
filtered_p, p_n = apply_top_p(PROBS, COMPARE_P)

sorted_idx3 = np.argsort(PROBS)[::-1]
snames = [TOKENS[i] for i in sorted_idx3]
sorig  = PROBS[sorted_idx3]
sk     = filtered_k[sorted_idx3]
sp     = filtered_p[sorted_idx3]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'🎮 Playground: top-k={K_VALUE}  vs  top-p={COMPARE_P}',
             fontsize=14, fontweight='bold', color='#ffd93d')

x = np.arange(10)
axes[0].bar(x, sorig, width=0.7, color='#8b949e', alpha=0.7, label='Original')
axes[0].bar(x, sk,   width=0.7, color='#ffd93d', alpha=0.9, label=f'top-k={K_VALUE}')
axes[0].axvline(K_VALUE-0.5, color='#ffd93d', lw=2.5, ls='--')
axes[0].set_xticks(x); axes[0].set_xticklabels(snames, rotation=45, ha='right', fontsize=8)
axes[0].set_title(f'top-k={K_VALUE}\n{K_VALUE} tokens in nucleus', color='#ffd93d', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_ylim(0, max(sorig)*1.2)

axes[1].bar(x, sorig, width=0.7, color='#8b949e', alpha=0.7, label='Original')
axes[1].bar(x, sp,   width=0.7, color='#a8e6cf', alpha=0.9, label=f'top-p={COMPARE_P}')
axes[1].axvline(p_n-0.5, color='#a8e6cf', lw=2.5, ls='--')
axes[1].set_xticks(x); axes[1].set_xticklabels(snames, rotation=45, ha='right', fontsize=8)
axes[1].set_title(f'top-p={COMPARE_P}\n{p_n} tokens in nucleus', color='#a8e6cf', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].set_ylim(0, max(sorig)*1.2)

# Difference chart
diff = sp - sk
cols_diff = ['#a8e6cf' if d > 0 else '#ff6b9d' for d in diff]
axes[2].bar(x, diff, width=0.7, color=cols_diff, alpha=0.9)
axes[2].axhline(0, color='white', lw=1)
axes[2].set_xticks(x); axes[2].set_xticklabels(snames, rotation=45, ha='right', fontsize=8)
axes[2].set_title('Difference (top-p minus top-k)\nGreen = top-p gives MORE, Red = LESS',
                  color='#ffd93d', fontweight='bold')

ent_k = -np.sum(sk * np.log(sk + 1e-12))
ent_p = -np.sum(sp * np.log(sp + 1e-12))
print(f"📊 Comparison:")
print(f"   top-k={K_VALUE}:         {K_VALUE} tokens,  entropy = {ent_k:.3f} bits")
print(f"   top-p={COMPARE_P}: {p_n} tokens,  entropy = {ent_p:.3f} bits")
print(f"   Difference:    {'top-p has more diversity' if ent_p > ent_k else 'top-k has more diversity'}")

plt.tight_layout()
plt.savefig('graph4_topk_playground.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## ✅ Notebook 3 Complete

| Concept | In One Line |
|---|---|
| Top-k | Keep exactly k highest-ranked tokens; discard the rest |
| k=1 | Greedy decoding — same token every time |
| Fixed | Nucleus size never changes, regardless of distribution shape |
| Failure (peaked) | Wastes slots on near-zero tokens |
| Failure (flat) | Cuts tokens that genuinely have meaningful probability |

> **🎯 Bouncer metaphor holds:** The list has exactly k names. Token #k+1 never gets in, no matter how high their probability.

**Next → `04_combined.ipynb`: The Full Recipe 🍳**
